# Lab Assignment 1 — Pandas vs. Polars
**ARTI 303 — Programming for AI**

| Team Members | Student ID |
|---|---|
| Abdullah AlAhmed (Leader) | 2250006144 |
| Fawaz AlOtaibi | 2250002441 |
| Nasser AlSaleh | 2250002475 |
| Tamim AlOmair | 2250007177 |
| Saleh Al Mulhim | 2250001665 |

**Section:** _[add your section]_  
**Date submitted:** 9 October 2026

---

## What this assignment is about

Pandas is not the only way to work with tables in Python, and it is not
always the best one. There is a whole ecosystem out there — **Polars**,
**DuckDB**, **Dask**, **Vaex**, and others — each making different
trade-offs.

The point of this assignment is not to prove one library beats another.
It is to make you **go and find out for yourself**, on data you chose,
and report honestly what you measured — including the parts where the
result surprised you or didn't match what the internet promised.

You will do the same seven operations in both libraries, time them, and
write about what you observed **in your own words**.

---

## What you will submit

**1. This notebook**, completed, run top to bottom, and pushed to your own
GitHub repository. Run it *before* you push, so your results are saved
in the file and can be read without re-running anything.

**2. A Word document** containing, in this order:

1. Your name, student ID, and section
2. A screenshot of you **pulling** this assignment
3. Your dataset link
4. Your GitHub repository link
5. A screenshot of you **pushing** your completed work

### The two screenshots

Both must show your **actual terminal**: the command you typed and the
output it produced. A screenshot of the GitHub website is not accepted.


> Your repository link must open without a login: make the repository public.

---
# Part 1 · Setup

Install Polars into your course environment, with `(.venv)` active in your terminal, then run the cell below.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import time
import statistics

print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.5
polars: 2.0.0


---
# Part 2 · Load your dataset

Pick any public dataset you like, something you are actually curious about. Requirements:

- **At least 100,000 rows.** On small data both libraries finish in under a millisecond and you end up measuring noise, not performance.
- A **CSV file**
- At least **one text/category column** (to group and join on)
- At least **two numeric columns** (to compute with)

Kaggle, data.gov.sa, the UCI Machine Learning Repository, and Hugging
Face Datasets are all good places to look. If the file is over about
50 MB, do not commit it: add it to `.gitignore` and put the download
link in your Word document.

Set `DATA_PATH` to your file, then run the cell.

In [2]:
DATA_PATH = "data/flights-3m.csv"

df_pandas = pd.read_csv(DATA_PATH)
df_polars = pl.read_csv(DATA_PATH)

print("pandas shape:", df_pandas.shape)
print("polars shape:", df_polars.shape)
print()
print("columns:", list(df_pandas.columns))
print()
print(df_pandas.head(3))

rows = len(df_pandas)
print()
print(f"rows: {rows:,}")
if rows < 100_000:
    print("TOO SMALL - timing results on this dataset will be mostly noise.")
    print("Pick a larger dataset before continuing.")
else:
    print("Size is fine.")

pandas shape: (231083, 5)
polars shape: (231083, 5)

columns: ['date', 'delay', 'distance', 'origin', 'destination']

      date  delay  distance origin destination
0  1010001     14       405    MCI         MDW
1  1010530    -11       370    LAX         PHX
2  1010540      5       389    ONT         SMF

rows: 231,083
Size is fine.


---
# Part 3 · How to time things properly


### Rule 1 — Run it more than once, take the median

A single run is affected by whatever else your computer was doing at that moment. Run each operation several times and take the middle value.

### Rule 2 — Polars is lazy. You must call `.collect()`

Polars has two modes. In **lazy** mode (`pl.scan_csv`, `.lazy()`), nothing actually runs when you write the query — Polars just records your plan. The work only happens when you call `.collect()`.

So if you time a lazy query without `.collect()`, you are timing the act of *writing down a plan*, not executing it. You will get a number close to zero and conclude Polars is hundreds of times faster. It is not. You measured nothing.

### Rule 3 — The first read of a file is slower

Your operating system caches file contents after the first read. Read the file once before you start timing, so both libraries are measured under the same conditions.

### The timing helper

Use this function for every measurement below.

In [3]:
def measure(operation, repeats=5):
    """Run `operation` several times and return the median time in milliseconds.

    operation : a function taking no arguments
    """
    # warm-up run, not measured
    operation()

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        operation()
        timings.append(time.perf_counter() - start)

    return statistics.median(timings) * 1000

### See Rule 2 for yourself

Run this and look at the difference.

In [4]:
lazy_no_collect = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()))
lazy_collected  = measure(lambda: pl.scan_csv(DATA_PATH).filter(pl.col(df_pandas.columns[0]).is_not_null()).collect())

print(f"lazy WITHOUT .collect(): {lazy_no_collect:8.3f} ms   <- nothing was executed")
print(f"lazy WITH .collect():    {lazy_collected:8.3f} ms   <- the real cost")

lazy WITHOUT .collect():    0.027 ms   <- nothing was executed
lazy WITH .collect():      16.767 ms   <- the real cost


**Record what you saw:**

| | ms |
|---|---|
| Lazy query without `.collect()` | about 0.03 ms (0.027) |
| Lazy query with `.collect()` | about 16.8 ms |

In one sentence — why is the first number meaningless?

Because without `.collect()` Polars only wrote down the plan and never opened or read the file, so I measured how long it takes to build the plan, not to run the query (it was roughly 600x smaller than the real cost).

---
# Part 4 · The seven operations

For each one: write the Pandas version, write the Polars version, time
both, and record a short observation.

Define which columns you are using once, here.

In [5]:
# Set these to real column names from YOUR dataset
CATEGORY_COL = "origin"    # a text/category column, for grouping
NUMERIC_COL  = "delay"     # a numeric column
NUMERIC_COL2 = "distance"  # a second numeric column

for col in [CATEGORY_COL, NUMERIC_COL, NUMERIC_COL2]:
    if col not in df_pandas.columns:
        print(f"[!] '{col}' is not in your dataset - update it above")
    else:
        print(f"[ok] {col}")

[ok] origin
[ok] delay
[ok] distance


## 4.1 — Reading the file

Time reading the whole file from disk in each library.

In [6]:
pandas_time = measure(lambda: pd.read_csv(DATA_PATH))
polars_time = measure(lambda: pl.read_csv(DATA_PATH))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

read_pandas, read_polars = pandas_time, polars_time

pandas:    93.91 ms
polars:    22.09 ms
ratio:      4.25x


**Observation:** Reading was one of the biggest gaps: Pandas took about 94 ms and Polars about 22 ms, so Polars was around 4x faster. The file is only 5.5 MB, so I think the difference comes from Polars parsing the CSV in parallel on several cores while `read_csv` in Pandas is mostly single-threaded.

## 4.2 — Filtering rows

Keep only rows where your numeric column is above its median.

In [7]:
threshold = df_pandas[NUMERIC_COL].median()
print("threshold:", threshold)

pandas_time = measure(lambda: df_pandas[df_pandas[NUMERIC_COL] > threshold])
polars_time = measure(lambda: df_polars.filter(pl.col(NUMERIC_COL) > threshold))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

filter_pandas, filter_polars = pandas_time, polars_time

threshold: 0.0
pandas:     4.89 ms
polars:     1.63 ms
ratio:      3.00x


**Observation:** Both are very fast (about 4.9 ms for Pandas, about 1.6 ms for Polars), so Polars was about 3x faster, but in absolute terms it is only a difference of 3 ms. With only 231,083 rows and one simple condition there is not much work, so a ratio like this is easy to change between runs (in my first test run the same filter was only about 1.6x).

## 4.3 — Creating a new column

Compute a new column from two existing ones.

In [8]:
pandas_time = measure(
    lambda: df_pandas.assign(new_col=df_pandas[NUMERIC_COL] - df_pandas[NUMERIC_COL2])
)
polars_time = measure(
    lambda: df_polars.with_columns(
        (pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col")
    )
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

newcol_pandas, newcol_polars = pandas_time, polars_time

pandas:     0.90 ms
polars:     0.56 ms
ratio:      1.61x


**Observation:** Both were under 1 ms (Pandas about 0.9 ms, Polars about 0.56 ms, ratio about 1.6x), so this is the smallest real difference together with sorting. Subtracting two columns is a simple vectorized operation, and Pandas already does it with fast NumPy code. At this size the numbers are mostly noise (in my first run the ratio was 1.03x).

## 4.4 — Group by and aggregate

Group by your category column and compute a sum and a mean.

In [9]:
pandas_time = measure(
    lambda: df_pandas.groupby(CATEGORY_COL).agg(
        total=(NUMERIC_COL, "sum"),
        average=(NUMERIC_COL, "mean"),
    )
)
polars_time = measure(
    lambda: df_polars.group_by(CATEGORY_COL).agg([
        pl.col(NUMERIC_COL).sum().alias("total"),
        pl.col(NUMERIC_COL).mean().alias("average"),
    ])
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

groupby_pandas, groupby_polars = pandas_time, polars_time

# Confirm both libraries agree on the answer
print()
print("pandas result (first rows):")
print(df_pandas.groupby(CATEGORY_COL)[NUMERIC_COL].sum().head())
print()
print("polars result (first rows):")
print(df_polars.group_by(CATEGORY_COL).agg(pl.col(NUMERIC_COL).sum()).head())

pandas:    27.05 ms
polars:     7.61 ms
ratio:      3.55x

pandas result (first rows):
origin
ABQ    38911
ALB     2303
AMA     6724
AUS    21966
BDL     9170
Name: delay, dtype: int64

polars result (first rows):
shape: (5, 2)
┌────────┬────────┐
│ origin ┆ delay  │
│ ---    ┆ ---    │
│ str    ┆ i64    │
╞════════╪════════╡
│ IND    ┆ 9659   │
│ LAX    ┆ 124646 │
│ SLC    ┆ 22494  │
│ SAT    ┆ 21124  │
│ LAS    ┆ 151429 │
└────────┴────────┘


**Observation:** Polars was about 3.5x faster (roughly 27 ms vs 7.6 ms). Both split the rows by airport code and sum/average `delay`, but Polars does the groups in parallel.

**Do both libraries give the same numbers?** Yes, the totals match (for example ALB = 2303 in both). The row order is different: Pandas sorts the groups alphabetically by `origin` (ABQ, ALB, AMA ...), but Polars returns them in no fixed order, so if I wanted to compare them row by row I would have to sort first.

## 4.5 — Sorting and taking the top N

In [10]:
pandas_time = measure(
    lambda: df_pandas.sort_values(NUMERIC_COL, ascending=False).head(10)
)
polars_time = measure(
    lambda: df_polars.sort(NUMERIC_COL, descending=True).head(10)
)

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

sort_pandas, sort_polars = pandas_time, polars_time

pandas:    18.54 ms
polars:    12.26 ms
ratio:      1.51x


**Observation:** Polars was a bit faster (about 12 ms vs 18.5 ms, around 1.5x), which was the smallest ratio of all seven operations. Both have to order all 231k values to find the top 10, so most of the work is the same and the gap is small.

## 4.6 — The Lazy API

This one has no Pandas equivalent, which is the point. Pandas executes every step immediately. Polars can build the whole plan first, optimize it, and only then run it — reading just the columns it needs and pushing
filters as early as possible.

Chain several operations together lazily, then `.collect()`.

In [11]:
def polars_lazy_pipeline():
    return (
        pl.scan_csv(DATA_PATH)
          .filter(pl.col(NUMERIC_COL) > threshold)
          .with_columns((pl.col(NUMERIC_COL) - pl.col(NUMERIC_COL2)).alias("new_col"))
          .group_by(CATEGORY_COL)
          .agg(pl.col("new_col").sum().alias("total"))
          .sort("total", descending=True)
          .collect()
    )

def pandas_equivalent():
    d = pd.read_csv(DATA_PATH)
    d = d[d[NUMERIC_COL] > threshold]
    d = d.assign(new_col=d[NUMERIC_COL] - d[NUMERIC_COL2])
    d = d.groupby(CATEGORY_COL).agg(total=("new_col", "sum"))
    return d.sort_values("total", ascending=False)

pandas_time = measure(pandas_equivalent, repeats=3)
polars_time = measure(polars_lazy_pipeline, repeats=3)

print(f"pandas (step by step): {pandas_time:8.2f} ms")
print(f"polars (lazy + collect): {polars_time:8.2f} ms")
print(f"ratio: {pandas_time/polars_time:8.2f}x")

lazy_pandas, lazy_polars = pandas_time, polars_time

pandas (step by step):    98.57 ms
polars (lazy + collect):    15.49 ms
ratio:     6.36x


**Look at the query plan.** Polars can show you what it decided to do.

In [12]:
plan = (
    pl.scan_csv(DATA_PATH)
      .filter(pl.col(NUMERIC_COL) > threshold)
      .group_by(CATEGORY_COL)
      .agg(pl.col(NUMERIC_COL).sum())
)

print(plan.explain())

AGGREGATE[maintain_order: false]
  [col("delay").sum()] BY [col("origin")]
  FROM
  simple π 2/2 ["origin", "delay"]
    Csv SCAN [data/flights-3m.csv]
    PROJECT 2/5 COLUMNS
    SELECTION: col("delay").cast(Float64) > 0.0
    ESTIMATED ROWS: 230647


**Observation:** The plan shows that Polars scans the CSV and reads only **2 of the 5 columns** (`PROJECT 2/5 COLUMNS`: just `origin` and `delay`), and the filter (`SELECTION: delay > 0`) is applied inside the scan itself, so rows are removed while reading instead of after loading everything. Pandas version loads all 5 columns and all rows first and filters later, which is one reason the chained pipeline was the biggest speedup (about 6x).

## 4.7 — Joining two tables

Build a small lookup table from your data, then join it back.

In [13]:
# a lookup table: average value per category
lookup_pandas = df_pandas.groupby(CATEGORY_COL, as_index=False)[NUMERIC_COL].mean()
lookup_pandas = lookup_pandas.rename(columns={NUMERIC_COL: "category_average"})

lookup_polars = (
    df_polars.group_by(CATEGORY_COL)
             .agg(pl.col(NUMERIC_COL).mean().alias("category_average"))
)

pandas_time = measure(lambda: df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left"))
polars_time = measure(lambda: df_polars.join(lookup_polars, on=CATEGORY_COL, how="left"))

print(f"pandas: {pandas_time:8.2f} ms")
print(f"polars: {polars_time:8.2f} ms")
print(f"ratio:  {pandas_time/polars_time:8.2f}x")

join_pandas, join_polars = pandas_time, polars_time

# Row-count check - a join should not change the number of rows here
merged = df_pandas.merge(lookup_pandas, on=CATEGORY_COL, how="left")
print()
print("rows before:", len(df_pandas), "-> after:", len(merged))
print("row count unchanged:", len(merged) == len(df_pandas))

pandas:    19.88 ms
polars:     4.10 ms
ratio:      4.84x



rows before: 231083 -> after: 231083
row count unchanged: True


**Observation:** The join was one of the largest differences: Pandas about 20 ms vs Polars about 4 ms, around 4.8x. The row count stayed 231,083 before and after, which is what I expected for a left join on a lookup table with one row per airport.

---
# Part 5 · Results summary

Collect every measurement into one table.

In [14]:
results = pd.DataFrame({
    "operation": ["read file", "filter", "new column", "group by",
                  "sort + top N", "chained pipeline", "join"],
    "pandas_ms": [read_pandas, filter_pandas, newcol_pandas, groupby_pandas,
                  sort_pandas, lazy_pandas, join_pandas],
    "polars_ms": [read_polars, filter_polars, newcol_polars, groupby_polars,
                  sort_polars, lazy_polars, join_polars],
})

results["speedup"] = (results["pandas_ms"] / results["polars_ms"]).round(2)
results["faster"] = np.where(results["speedup"] > 1, "polars", "pandas")

results.round(2)

,operation,pandas_ms,polars_ms,speedup,faster
0,read file,93.91,22.09,4.25,polars
1,filter,4.89,1.63,3.00,polars
2,new column,0.90,0.56,1.61,polars
3,group by,27.05,7.61,3.55,polars
4,sort + top N,18.54,12.26,1.51,polars
5,chained pipeline,98.57,15.49,6.36,polars
6,join,19.88,4.10,4.84,polars


In [15]:
# Save your results so they are in the repository as evidence
import os
os.makedirs("results", exist_ok=True)
results.round(3).to_csv("results/benchmark.csv", index=False)
print("saved to results/benchmark.csv")

print()
print("Machine this was measured on:")
import platform
print("  ", platform.platform())
print("  ", platform.processor() or "processor not reported")

saved to results/benchmark.csv

Machine this was measured on:
   Linux-6.18.44-fc-v80-x86_64-with-glibc2.39
   x86_64


> Recording the machine matters. Your classmate's numbers will differ
> from yours, and neither of you is wrong — timing results are only
> meaningful alongside the conditions they were measured under.

---
# Part 6 · Your analysis

### Q1 — Where was the difference largest, and where was it smallest?

The largest differences were the chained pipeline (about 6.4x: 98.6 ms vs 15.5 ms), the join (about 4.8x: 19.9 ms vs 4.1 ms) and reading the file (about 4.3x: 93.9 ms vs 22.1 ms). The smallest were sorting + top N (about 1.5x: 18.5 ms vs 12.3 ms) and creating a new column (about 1.6x, both under 1 ms). Filter (3.0x) and group by (3.6x) were in the middle. My theory is that Polars wins most where there is a lot of work that can be split across cores (parsing the CSV, hashing keys for a join) or where it can skip work (the lazy plan only reads 2 of 5 columns). It wins least where the operation is one simple pass over a column or both libraries must do the same full sort, because Pandas already uses fast compiled code for that.

### Q2 — Did anything surprise you?

Yes, two things. First, how tiny the lazy time was without `.collect()` (0.027 ms vs about 16.8 ms), which would make Polars look about 600x faster if I forgot to collect. Second, the results changed between runs: in my first test run the filter ratio was 1.6x and the new column ratio was 1.03x, but in the final run they were 3.0x and 1.6x. Also the group-by gave the same totals but in a different row order. So the small operations (under 5 ms) are not very reliable.

### Q3 — What was harder to write?

For simple things like filtering and sorting, Pandas syntax felt shorter and more familiar (`df[df[col] > x]`). Polars needs `pl.col(...)` and `.alias(...)`, which felt longer at first, but for the group-by and the lazy pipeline the Polars version was actually easier to read because the steps are chained in order. The part I had to think about most was the lazy API and remembering to call `.collect()`, and I had to read the query plan output to understand what it was telling me.

### Q4 — The marketing claim

Only partly. Only one operation, the chained pipeline (6.36x), was above 5x; the join (4.84x) and reading the file (4.25x) came close, but filter (3.0x), group by (3.55x), new column (1.61x) and sort (1.51x) were well below it. I think the published benchmarks usually use much bigger datasets (millions or billions of rows), more CPU cores, and heavy operations like big joins and group-bys, where parallelism and the optimizer matter more. My dataset has only 231k rows and is 5.5 MB, so the work is small and a few milliseconds of overhead is a large part of the time. Also the results depend on the machine I ran it on (listed in Part 5) and on the median of only 5 runs (3 for the pipeline).

### Q5 — Your recommendation

I would choose Polars for a project with big files or heavy pipelines (many joins, group-bys and filters, like processing millions of rows of logs or sales data), where speed and memory matter, especially because of the lazy mode. I would choose Pandas for small or medium data, quick exploration in a notebook, or when the project depends on other libraries that expect a Pandas DataFrame (like scikit-learn and many plotting libraries) and the team already knows it, because on small data the speed difference is only a few milliseconds.

### Q6 — What else is out there?

**DuckDB** — documentation: https://duckdb.org/docs/. It is an in-process analytical database that lets you run SQL queries directly on CSV/Parquet files or on DataFrames, without setting up a database server. It exists to make fast analytics on local data easy, including data that is larger than memory.

---
# Submission checklist

- [ ] Part 1: `polars` installed and recorded in `requirements.txt`
- [ ] Part 2: dataset loaded, with 100,000+ rows
- [ ] Part 3: lazy `.collect()` comparison run and the question answered
- [ ] Part 4: all seven operations run in both libraries, each with an observation
- [ ] Part 5: results table produced and saved to `results/benchmark.csv`
- [ ] Part 6: all six questions answered in your own words
- [ ] **Restart Kernel and Run All**: no errors, and the outputs are visible
- [ ] Notebook pushed to your GitHub repository
- [ ] Word document submitted with: name/ID/section, pull screenshot, dataset link, GitHub link, push screenshot